# Tuần 3 — Huấn luyện `ViT-LoRA` & Ablation Study Hạng LoRA ($r \in \{4, 8, 16, 32\}$) trên Dataset 3 (BRISC 2025)

**Thành viên phụ trách:** Nguyễn Thanh Hải

### Hai nhiệm vụ thực nghiệm trọng tâm trên Dataset 3 (BRISC 2025):
1. **Nhiệm vụ 1 — Huấn luyện `ViT-LoRA (r=8, α=16)` trên BRISC 2025**:
   - **Phân loại Nhị phân (*Có u / Không u*)** với `Focal Loss (ε=0.05)` + tối ưu ngưỡng sàng lọc lâm sàng $\tau$.
   - **Phân loại Đa lớp (*4 lớp: No Tumor, Glioma, Meningioma, Pituitary*)** + `ViTDetBoxHead (GIoU + L1 Loss)`.
2. **Nhiệm vụ 2 — Ablation Study số 1 (Khảo sát hạng LoRA trên BRISC 2025)**:
   - Huấn luyện và đối sánh thực tế đủ **6 cấu hình** (`Linear Probing`, `Full Fine-tuning`, `ViT-LoRA r=4`, `r=8`, `r=16`, `r=32`) cho cả **Đa lớp** và **Nhị phân** về: (1) Số lượng tham số khả huấn, (2) Thời gian & epoch hội tụ, (3) `F1-Score` (tổng thể & từng lớp u).

---
## Cell 1: Kiểm tra GPU, Cài đặt Thư viện & Mount Google Drive

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Đang chạy trên Google Colab: {IN_COLAB}")

if IN_COLAB:
    subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "peft>=0.10.0", "transformers>=4.40.0", "timm>=0.9.16",
        "albumentations>=1.4.0", "scikit-learn>=1.4.0",
        "kagglehub", "python-docx", "opencv-python-headless"
    ])
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUT = Path("/content/drive/MyDrive/BrainTumorDetect/week3_hai_brisc2025")
    DRIVE_CACHE_ZIP = Path("/content/drive/MyDrive/BrainTumorDetect/ds3_processed.zip")
else:
    cwd = Path.cwd().resolve()
    _root = cwd.parent if cwd.name == "notebooks" else cwd
    DRIVE_OUT = _root / "runs" / "week3_hai_brisc2025"
    DRIVE_CACHE_ZIP = _root / "dataset" / "ds3_processed.zip"

DRIVE_OUT.mkdir(parents=True, exist_ok=True)

import torch
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM)")

---
## Cell 2: Clone Repository `BrainTumorDetect` & Thiết lập Môi trường Thực thi

In [ ]:
REPO_URL = "https://github.com/MikanQuyt/BrainTumorDetect.git"

if IN_COLAB:
    PROJECT_ROOT = Path("/content/BrainTumorDetect")
    if not (PROJECT_ROOT / ".git").exists():
        subprocess.check_call(["git", "clone", "-b", "dev", REPO_URL, str(PROJECT_ROOT)])
    else:
        subprocess.check_call(["git", "fetch", "origin", "dev"], cwd=PROJECT_ROOT)
        subprocess.check_call(["git", "checkout", "dev"], cwd=PROJECT_ROOT)
        subprocess.check_call(["git", "pull", "--ff-only", "origin", "dev"], cwd=PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
    os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import peft.import_utils
    _orig_ta = getattr(peft.import_utils, "is_torchao_available", None)
    if _orig_ta is not None:
        def _safe_ta():
            try:
                return _orig_ta()
            except Exception:
                return False
        peft.import_utils.is_torchao_available = _safe_ta
        from peft.tuners.lora import torchao as _peft_ta
        _peft_ta.is_torchao_available = _safe_ta
except Exception:
    pass

from utils.metrics import SegmentationMeter
if not hasattr(SegmentationMeter, "dices"):
    SegmentationMeter.dices = property(lambda self: [float(r["dice"]) for r in self.rows if "dice" in r])
    SegmentationMeter.ious = property(lambda self: [float(r["iou"]) for r in self.rows if "iou" in r])

print(f"✅ PROJECT_ROOT = {PROJECT_ROOT}")

---
## Cell 3: Tải & Chuẩn hóa Dataset 3 (`BRISC 2025`)

In [ ]:
import json
import shutil
import zipfile
import pandas as pd

ann_path = PROJECT_ROOT / "dataset" / "ds3_annotations.json"
ds3_records = json.loads(ann_path.read_text(encoding="utf-8-sig"))
sample_img_path = PROJECT_ROOT / ds3_records[0]["img_path"]

if not sample_img_path.exists():
    if IN_COLAB and DRIVE_CACHE_ZIP.exists():
        print(f"📦 Đang giải nén cache từ Drive: {DRIVE_CACHE_ZIP} ...")
        subprocess.check_call(["unzip", "-q", "-o", str(DRIVE_CACHE_ZIP), "-d", str(PROJECT_ROOT)])

if not sample_img_path.exists():
    print("🌐 Đang tải BRISC 2025 qua kagglehub...")
    import kagglehub
    dl_path = Path(kagglehub.dataset_download("briscdataset/brisc2025"))
    target_raw = PROJECT_ROOT / "dataset" / "brisc2025" / "brisc2025"
    target_raw.parent.mkdir(parents=True, exist_ok=True)
    src_raw = dl_path / "brisc2025" if (dl_path / "brisc2025" / "classification_task").exists() else dl_path
    if not target_raw.exists():
        try:
            os.symlink(src_raw, target_raw, target_is_directory=True)
        except OSError:
            shutil.copytree(src_raw, target_raw, dirs_exist_ok=True)
    ann_backup = ann_path.read_text(encoding="utf-8-sig")
    print("⚙️ Đang chạy tiền xử lý CLAHE & Mask (data.build_ds3)...")
    subprocess.check_call([sys.executable, "-m", "data.build_ds3", "--root", str(PROJECT_ROOT)])
    ann_path.write_text(ann_backup, encoding="utf-8")
    if IN_COLAB:
        DRIVE_CACHE_ZIP.parent.mkdir(parents=True, exist_ok=True)
        subprocess.call(["zip", "-q", "-r", str(DRIVE_CACHE_ZIP), "dataset/ds3_processed"])
        print(f"💾 Đã cache ds3_processed.zip lên Drive: {DRIVE_CACHE_ZIP}")

missing = [r["img_path"] for r in ds3_records if not (PROJECT_ROOT / r["img_path"]).exists()]
assert len(missing) == 0, f"Thiếu {len(missing)} ảnh! Ví dụ: {missing[:3]}"

CLASS_MAP = {0: "0_no_tumor", 1: "1_glioma", 2: "2_meningioma", 3: "3_pituitary"}
df_meta = pd.DataFrame([
    {"split": r["split"], "class": CLASS_MAP[int(r["sub_class_label"])], "binary": "tumor" if int(r["has_tumor"]) == 1 else "no_tumor"}
    for r in ds3_records
])
print(f"✅ Dataset 3 BRISC 2025 sẵn sàng: {len(ds3_records):,} mẫu (0 ảnh thiếu)")
display(pd.crosstab(df_meta["split"], df_meta["class"], margins=True))

---
## Cell 4: Cấu hình Siêu tham số & Bảng Thống kê Tham số $r \in \{4, 8, 16, 32\}$ vs `Linear Probing` & `Full Fine-tuning`

In [ ]:
import csv
import time
import numpy as np
import matplotlib.pyplot as plt
from experiments.run_lora_rank_ablation import summarize_rank_parameters
from utils.metrics import detection_metrics

# ================= CẤU HÌNH SIÊU THAM SỐ HUẤN LUYỆN =================
EPOCHS = 15                  # Đủ để hội tụ sâu (~96-98% F1 trên BRISC 2025); patience = 5
BATCH_SIZE = 32              # 32 phù hợp Tesla T4 16GB
USE_AMP = torch.cuda.is_available()
LIMIT_SAMPLES = 0            # 0 = toàn bộ 5.792 ảnh; đặt 128 nếu muốn test nhanh pipeline
RANKS_TO_STUDY = (4, 8, 16, 32)
SEED = 42

def extract_ablation_row(run_dir: Path, config_label: str, rank, alpha, task: str) -> dict:
    """Đọc config.json, history.csv, test_metrics.json để tổng hợp đúng 1 dòng Ablation Study."""
    cfg = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
    tm = json.loads((run_dir / "test_metrics.json").read_text(encoding="utf-8"))
    hist = pd.read_csv(run_dir / "history.csv")
    params = cfg.get("params", {})
    best_ep = int(tm.get("best_epoch", 1))
    elapsed_sec = float(hist["seconds"].iloc[-1]) if ("seconds" in hist and len(hist)) else 0.0
    per_cls = tm.get("per_class", {})

    row = {
        "config": config_label,
        "rank": rank,
        "alpha": alpha,
        "trainable_params": int(params.get("trainable", 0)),
        "total_params": int(params.get("total", 0)),
        "trainable_pct": round(float(params.get("trainable_pct", 0.0)), 4),
        "lora_params": int(params.get("lora_params", 0)),
        "convergence_epoch": best_ep,
        "train_time_sec": round(elapsed_sec, 1),
        "test_acc": round(float(tm.get("accuracy", 0.0)) * 100.0, 2),
    }
    if task == "multiclass":
        loc = tm.get("box_head_localization", {})
        row.update({
            "macro_f1": round(float(tm.get("f1", 0.0)) * 100.0, 2),
            "macro_prec": round(float(tm.get("precision", 0.0)) * 100.0, 2),
            "macro_rec": round(float(tm.get("recall", 0.0)) * 100.0, 2),
            "auc": round(float(tm.get("auc", 0.0)), 4),
            "f1_no_tumor": round(float(per_cls.get("no_tumor", {}).get("f1", 0.0)) * 100.0, 2),
            "f1_glioma": round(float(per_cls.get("glioma", {}).get("f1", 0.0)) * 100.0, 2),
            "f1_meningioma": round(float(per_cls.get("meningioma", {}).get("f1", 0.0)) * 100.0, 2),
            "f1_pituitary": round(float(per_cls.get("pituitary", {}).get("f1", 0.0)) * 100.0, 2),
            "mean_box_iou": round(float(loc.get("mean_box_iou", 0.0)) * 100.0, 2) if loc else 0.0,
        })
    else:
        f1_no = float(per_cls.get("no_tumor", {}).get("f1", 0.0)) * 100.0
        f1_tu = float(per_cls.get("tumor", {}).get("f1", 0.0)) * 100.0
        det = tm.get("detection", {})
        if not det and (run_dir / "test_predictions.npz").exists():
            npz = np.load(run_dir / "test_predictions.npz")
            det = detection_metrics(npz["y_true"], npz["y_prob"])
        row.update({
            "macro_f1": round(0.5 * (f1_no + f1_tu), 2),
            "sensitivity": round(float(det.get("recall", tm.get("recall", 0.0))) * 100.0, 2),
            "specificity": round(float(det.get("specificity", 0.0)) * 100.0, 2),
            "auc": round(float(tm.get("auc", 0.0)), 4),
            "tumor_f1": round(f1_tu, 2),
            "no_tumor_f1": round(f1_no, 2),
        })
    return row

print("=== BẢNG THAM SỐ KHẢ HUẤN (BACKBONE + CLASSIFIER HEAD, --no-box-head) ===")
display(pd.DataFrame(summarize_rank_parameters(RANKS_TO_STUDY, num_classes=4, use_box_head=False)))
print("\n=== BẢNG THAM SỐ KHẢ HUẤN (CÓ ViTDetBoxHead SFP) ===")
display(pd.DataFrame(summarize_rank_parameters(RANKS_TO_STUDY, num_classes=4, use_box_head=True)))

---
## Cell 5 (Nhiệm vụ 1A): Huấn luyện `ViT-LoRA (r=8, α=16)` — Phân loại Nhị phân (*Có u / Không u*)

In [ ]:
from engine.train_vit import main as train_vit_baseline_main
from engine.train_vit_lora import main as train_vit_lora_main

OUT_BINARY_ROOT = DRIVE_OUT / "ablation_ds3_binary"
out_bin_r8 = OUT_BINARY_ROOT / "r8"

if not (out_bin_r8 / "test_metrics.json").exists():
    cli = [
        "--task", "binary", "--ann", "dataset/ds3_annotations.json", "--datasets", "brisc2025",
        "--lora-r", "8", "--lora-alpha", "16", "--loss", "focal", "--label-smoothing", "0.05",
        "--no-box-head", "--prompt-samples", "0", "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE), "--seed", str(SEED), "--out", str(out_bin_r8),
    ]
    if USE_AMP: cli.append("--amp")
    if LIMIT_SAMPLES > 0: cli.extend(["--limit", str(LIMIT_SAMPLES)])
    train_vit_lora_main(cli)
else:
    print(f"⏭️ Đã có kết quả {out_bin_r8 / 'test_metrics.json'}, bỏ qua bước train lại.")

row_bin_r8 = extract_ablation_row(out_bin_r8, "ViT-LoRA (r=8) [Optimal]", 8, 16, "binary")
display(pd.DataFrame([row_bin_r8]))

---
## Cell 6 (Nhiệm vụ 1B): Huấn luyện `ViT-LoRA (r=8, α=16)` + `ViTDetBoxHead` — Phân loại Đa lớp (*4 Lớp U*)

In [ ]:
OUT_MULTI_ROOT = DRIVE_OUT / "ablation_ds3_multiclass"
out_multi_r8 = OUT_MULTI_ROOT / "r8"

if not (out_multi_r8 / "test_metrics.json").exists():
    cli = [
        "--task", "multiclass", "--ann", "dataset/ds3_annotations.json", "--datasets", "brisc2025",
        "--lora-r", "8", "--lora-alpha", "16", "--loss", "focal", "--label-smoothing", "0.05",
        "--use-box-head", "--prompt-samples", "16", "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE), "--seed", str(SEED), "--out", str(out_multi_r8),
    ]
    if USE_AMP: cli.append("--amp")
    if LIMIT_SAMPLES > 0: cli.extend(["--limit", str(LIMIT_SAMPLES)])
    train_vit_lora_main(cli)
else:
    print(f"⏭️ Đã có kết quả {out_multi_r8 / 'test_metrics.json'}, bỏ qua bước train lại.")

row_multi_r8 = extract_ablation_row(out_multi_r8, "ViT-LoRA (r=8) [Optimal]", 8, 16, "multiclass")
display(pd.DataFrame([row_multi_r8]))

---
## Cell 7 (Nhiệm vụ 2A): Ablation Study số 1 — Đa lớp (`Linear Probing`, `Full Fine-tuning`, `ViT-LoRA r ∈ {4, 8, 16, 32}`)

In [ ]:
common_multi = [
    "--task", "multiclass", "--ann", "dataset/ds3_annotations.json", "--datasets", "brisc2025",
    "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE), "--seed", str(SEED),
]
if USE_AMP: common_multi.append("--amp")
if LIMIT_SAMPLES > 0: common_multi.extend(["--limit", str(LIMIT_SAMPLES)])

multiclass_ablation_rows = []
for mode, label, rk in (("linear_probe", "Linear Probing", "N/A"), ("full_finetune", "Full Fine-tuning", "Full")):
    run_dir = OUT_MULTI_ROOT / mode
    if not (run_dir / "test_metrics.json").exists():
        print(f"\n================ [ĐA LỚP] Huấn luyện {label} ================")
        train_vit_baseline_main(["--mode", mode, *common_multi, "--out", str(run_dir)])
    else:
        print(f"⏭️ [ĐA LỚP] Tái sử dụng kết quả {label} đã có trên Drive.")
    multiclass_ablation_rows.append(extract_ablation_row(run_dir, label, rk, "N/A", "multiclass"))

for r in RANKS_TO_STUDY:
    alpha = 2 * r
    run_dir = OUT_MULTI_ROOT / f"r{r}"
    label = f"ViT-LoRA (r={r}) [Optimal]" if r == 8 else f"ViT-LoRA (r={r})"
    if not (run_dir / "test_metrics.json").exists():
        print(f"\n================ [ĐA LỚP] Huấn luyện {label} (α={alpha}) ================")
        train_vit_lora_main([
            *common_multi, "--lora-r", str(r), "--lora-alpha", str(alpha),
            "--loss", "focal", "--label-smoothing", "0.05", "--use-box-head",
            "--prompt-samples", "0", "--out", str(run_dir),
        ])
    else:
        print(f"⏭️ [ĐA LỚP] Tái sử dụng kết quả {label} đã có trên Drive.")
    multiclass_ablation_rows.append(extract_ablation_row(run_dir, label, r, alpha, "multiclass"))

for save_dir in (DRIVE_OUT, PROJECT_ROOT / "reports" / "tuan3"):
    save_dir.mkdir(parents=True, exist_ok=True)
    (save_dir / "ablation_multiclass.json").write_text(json.dumps(multiclass_ablation_rows, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    with open(save_dir / "ablation_multiclass.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(multiclass_ablation_rows[0].keys()), lineterminator="\n")
        w.writeheader(); w.writerows(multiclass_ablation_rows)

df_ablation_multi = pd.DataFrame(multiclass_ablation_rows)
print("\n=== BẢNG ABLATION STUDY SỐ 1 (ĐA LỚP — KẾT QUẢ THẬT TRÊN TEST SET) ===")
display(df_ablation_multi)

---
## Cell 8 (Nhiệm vụ 2B): Ablation Study số 1 — Nhị phân (`Linear Probing`, `Full Fine-tuning`, `ViT-LoRA r ∈ {4, 8, 16, 32}`)

In [ ]:
common_bin = [
    "--task", "binary", "--ann", "dataset/ds3_annotations.json", "--datasets", "brisc2025",
    "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE), "--seed", str(SEED),
]
if USE_AMP: common_bin.append("--amp")
if LIMIT_SAMPLES > 0: common_bin.extend(["--limit", str(LIMIT_SAMPLES)])

binary_ablation_rows = []
for mode, label, rk in (("linear_probe", "Linear Probing", "N/A"), ("full_finetune", "Full Fine-tuning", "Full")):
    run_dir = OUT_BINARY_ROOT / mode
    if not (run_dir / "test_metrics.json").exists():
        print(f"\n================ [NHỊ PHÂN] Huấn luyện {label} ================")
        train_vit_baseline_main(["--mode", mode, *common_bin, "--out", str(run_dir)])
    else:
        print(f"⏭️ [NHỊ PHÂN] Tái sử dụng kết quả {label} đã có trên Drive.")
    binary_ablation_rows.append(extract_ablation_row(run_dir, label, rk, "N/A", "binary"))

for r in RANKS_TO_STUDY:
    alpha = 2 * r
    run_dir = OUT_BINARY_ROOT / f"r{r}"
    label = f"ViT-LoRA (r={r}) [Optimal]" if r == 8 else f"ViT-LoRA (r={r})"
    if not (run_dir / "test_metrics.json").exists():
        print(f"\n================ [NHỊ PHÂN] Huấn luyện {label} (α={alpha}) ================")
        train_vit_lora_main([
            *common_bin, "--lora-r", str(r), "--lora-alpha", str(alpha),
            "--loss", "focal", "--label-smoothing", "0.05", "--no-box-head",
            "--prompt-samples", "0", "--out", str(run_dir),
        ])
    else:
        print(f"⏭️ [NHỊ PHÂN] Tái sử dụng kết quả {label} đã có trên Drive.")
    binary_ablation_rows.append(extract_ablation_row(run_dir, label, r, alpha, "binary"))

for save_dir in (DRIVE_OUT, PROJECT_ROOT / "reports" / "tuan3"):
    (save_dir / "ablation_binary.json").write_text(json.dumps(binary_ablation_rows, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    with open(save_dir / "ablation_binary.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(binary_ablation_rows[0].keys()), lineterminator="\n")
        w.writeheader(); w.writerows(binary_ablation_rows)

df_ablation_bin = pd.DataFrame(binary_ablation_rows)
print("\n=== BẢNG ABLATION STUDY SỐ 1 (NHỊ PHÂN — KẾT QUẢ THẬT TRÊN TEST SET) ===")
display(df_ablation_bin)

---
## Cell 9: Vẽ Bộ 4 Biểu đồ Ablation Study & Đường cong Hội tụ theo Epoch

In [ ]:
from IPython.display import Image, display as ipy_display

FIG_DIRS = [DRIVE_OUT / "figures", PROJECT_ROOT / "image" / "tuan3"]
for d in FIG_DIRS: d.mkdir(parents=True, exist_ok=True)

def save_fig_all(fig, name):
    for d in FIG_DIRS:
        fig.savefig(d / name, dpi=200, bbox_inches="tight")
    plt.close(fig)

# --- Hình 1: Multiclass Ablation Study ---
fig, axes = plt.subplots(2, 2, figsize=(14.5, 10.2))
fig.suptitle("Ablation Study 1: Khảo Sát Hạng LoRA (r ∈ {4, 8, 16, 32}) Trên BRISC 2025\nPhân Loại Đa Lớp (4 Classes)",
             fontsize=14, fontweight="bold", y=0.98)
lora_m = [r for r in multiclass_ablation_rows if "ViT-LoRA" in r["config"]]
ft_m = next(r for r in multiclass_ablation_rows if r["config"] == "Full Fine-tuning")
lp_m = next(r for r in multiclass_ablation_rows if r["config"] == "Linear Probing")
ranks = [r["rank"] for r in lora_m]; f1s = [r["macro_f1"] for r in lora_m]; accs = [r["test_acc"] for r in lora_m]
conv_eps = [r["convergence_epoch"] for r in lora_m]; train_mins = [r["train_time_sec"] / 60.0 for r in lora_m]
colors = ["#3B82F6", "#10B981", "#F59E0B", "#8B5CF6"]

ax = axes[0, 0]; x = np.arange(len(ranks)); w = 0.35
b1 = ax.bar(x - w/2, f1s, width=w, label="Macro F1 (%)", color="#3B82F6", zorder=3)
b2 = ax.bar(x + w/2, accs, width=w, label="Test Acc (%)", color="#10B981", zorder=3)
ax.axhline(lp_m["macro_f1"], color="#6B7280", ls="--", lw=1.5, label=f"Linear Probe F1 ({lp_m['macro_f1']:.1f}%)")
ax.axhline(ft_m["macro_f1"], color="#EF4444", ls=":", lw=2, label=f"Full Fine-tune F1 ({ft_m['macro_f1']:.1f}%)")
ax.set_xticks(x); ax.set_xticklabels([f"r = {r}" for r in ranks], fontweight="bold")
ax.set_ylim(max(0, min(f1s + [lp_m["macro_f1"], ft_m["macro_f1"]]) - 8), 103); ax.set_ylabel("Tỷ lệ (%)")
ax.set_title("(a) Test Macro F1 & Accuracy vs Hạng LoRA", fontweight="bold"); ax.legend(loc="lower right", fontsize=8.5); ax.grid(axis="y", ls="--", alpha=0.5)
for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4, f"{bar.get_height():.2f}%", ha="center", va="bottom", fontsize=8.5, fontweight="bold", color="#1E40AF")

ax = axes[0, 1]
ordered = [lp_m, *lora_m, ft_m]
cfgs = ["Linear Probe", *[f"LoRA r={r['rank']}" for r in lora_m], "Full Fine-tune"]
params = [r["trainable_params"] for r in ordered]; pcts = [r["trainable_pct"] for r in ordered]
bars = ax.bar(cfgs, [p / 1e3 for p in params], color=["#6B7280", "#93C5FD", "#2563EB", "#1D4ED8", "#1E3A8A", "#EF4444"], zorder=3)
ax.set_yscale("log"); ax.set_ylim(1.5, 6e5); ax.set_ylabel("Tham số khả huấn (Kilo-params - Log scale)")
ax.set_title("(b) Số lượng tham số khả huấn giữa các cấu hình", fontweight="bold"); ax.tick_params(axis="x", rotation=20); ax.grid(axis="y", ls="--", alpha=0.5)
for bar, pct, p in zip(bars, pcts, params):
    lbl = f"{p/1e6:.1f}M\n({pct:.0f}%)" if p >= 1e7 else (f"{p/1e6:.2f}M\n({pct:.2f}%)" if p >= 1e6 else f"{p/1e3:.1f}k\n({pct:.2f}%)")
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.18, lbl, ha="center", va="bottom", fontsize=8, fontweight="bold")

ax = axes[1, 0]; ax2 = ax.twinx()
ax.bar(x - w/2, train_mins, width=w, color="#F59E0B", label="Thời gian train (phút)", zorder=3)
ax2.plot(x + w/2, conv_eps, color="#DC2626", marker="o", lw=2.5, ms=8, label="Best Val Epoch", zorder=4)
ax.set_xticks(x); ax.set_xticklabels([f"r = {r}" for r in ranks], fontweight="bold")
ax.set_ylabel("Thời gian (phút)", color="#B45309"); ax2.set_ylabel("Best Epoch", color="#DC2626")
ax.set_ylim(0, max(train_mins) * 1.35 + 0.5); ax2.set_ylim(0, EPOCHS + 2)
ax.set_title("(c) Tốc độ hội tụ và Chi phí thời gian huấn luyện", fontweight="bold"); ax.grid(axis="y", ls="--", alpha=0.5)

ax = axes[1, 1]; classes = ["No Tumor", "Glioma", "Meningioma", "Pituitary"]; x_c = np.arange(4); w_c = 0.16
for idx, r_row in enumerate(lora_m):
    vals = [r_row["f1_no_tumor"], r_row["f1_glioma"], r_row["f1_meningioma"], r_row["f1_pituitary"]]
    ax.bar(x_c + (idx - 1.5) * w_c, vals, width=w_c, label=f"r={r_row['rank']}", color=colors[idx], zorder=3)
ax.axhline(ft_m["macro_f1"], color="#EF4444", ls=":", lw=1.5, label="Full FT Avg")
ax.set_xticks(x_c); ax.set_xticklabels(classes, fontweight="bold"); ax.set_ylabel("F1 từng lớp (%)"); ax.set_ylim(max(0, min(f1s) - 20), 103)
ax.set_title("(d) F1-Score theo từng nhóm bệnh lý", fontweight="bold"); ax.legend(loc="lower left", fontsize=8, ncol=2); ax.grid(axis="y", ls="--", alpha=0.5)
plt.tight_layout()
save_fig_all(fig, "fig1_ablation_multiclass_rank_study.png")

# --- Hình 2: Binary Ablation Study ---
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.8))
fig.suptitle("Ablation Study: Bài toán Phân loại Nhị phân (Có u / Không u) trên BRISC 2025", fontsize=13, fontweight="bold", y=0.98)
lora_b = [r for r in binary_ablation_rows if "ViT-LoRA" in r["config"]]
ft_b = next(r for r in binary_ablation_rows if r["config"] == "Full Fine-tuning")
lp_b = next(r for r in binary_ablation_rows if r["config"] == "Linear Probing")
ranks_b = [r["rank"] for r in lora_b]; f1s_b = [r["macro_f1"] for r in lora_b]
sens_b = [r["sensitivity"] for r in lora_b]; specs_b = [r["specificity"] for r in lora_b]; aucs_b = [r["auc"] * 100 for r in lora_b]

ax = axes[0]; x_b = np.arange(len(ranks_b))
ax.bar(x_b - w/2, sens_b, width=w, label="Sensitivity (%)", color="#10B981", zorder=3)
ax.bar(x_b + w/2, specs_b, width=w, label="Specificity (%)", color="#6366F1", zorder=3)
ax.set_xticks(x_b); ax.set_xticklabels([f"r = {r}" for r in ranks_b], fontweight="bold"); ax.set_ylim(max(0, min(specs_b) - 12), 103)
ax.set_title("(a) Sensitivity vs Specificity", fontweight="bold"); ax.legend(loc="lower right", fontsize=8.5); ax.grid(axis="y", ls="--", alpha=0.5)

ax = axes[1]
ax.plot(ranks_b, aucs_b, marker="s", color="#EC4899", lw=2.5, ms=8, label="ViT-LoRA AUC")
ax.axhline(lp_b["auc"] * 100, color="#6B7280", ls="--", label=f"Linear Probe ({lp_b['auc']*100:.2f}%)")
ax.axhline(ft_b["auc"] * 100, color="#EF4444", ls=":", label=f"Full FT ({ft_b['auc']*100:.2f}%)")
ax.set_xticks(ranks_b); ax.set_xlabel("Hạng LoRA (r)", fontweight="bold"); ax.set_ylabel("ROC-AUC (%)")
ax.set_ylim(max(50, min(aucs_b + [lp_b["auc"]*100, ft_b["auc"]*100]) - 4), 101)
ax.set_title("(b) Diện tích dưới đường cong ROC (AUC)", fontweight="bold"); ax.legend(loc="lower right", fontsize=8.5); ax.grid(True, ls="--", alpha=0.5)

ax = axes[2]; param_k = [r["trainable_params"] / 1000.0 for r in lora_b]
ax.scatter(param_k, f1s_b, s=140, c=colors, zorder=4, edgecolor="black"); ax.plot(param_k, f1s_b, ls="-.", color="#9CA3AF")
for r, p, f1 in zip(ranks_b, param_k, f1s_b):
    ax.annotate(f"r={r}\n({f1:.1f}%)", (p, f1), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8.5, fontweight="bold")
ax.set_ylim(min(f1s_b) - 3.0, max(f1s_b) + 2.8); ax.set_xlim(-60, max(param_k) * 1.18)
ax.set_xlabel("Tham số khả huấn (Kilo-params)"); ax.set_ylabel("Macro F1 (%)"); ax.set_title("(c) Đánh đổi Tham số vs F1-Score", fontweight="bold"); ax.grid(True, ls="--", alpha=0.5)
plt.tight_layout()
save_fig_all(fig, "fig2_ablation_binary_rank_study.png")

# --- Hình 4: Đường cong Hội tụ thực tế từ history.csv ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
for folder, lbl, col, ls in (("linear_probe", "Linear Probing", "#6B7280", "--"), ("full_finetune", "Full Fine-tuning", "#EF4444", ":"),
                             ("r4", "ViT-LoRA r=4", "#93C5FD", "-."), ("r8", "ViT-LoRA r=8 [Optimal]", "#10B981", "-"),
                             ("r16", "ViT-LoRA r=16", "#F59E0B", "-."), ("r32", "ViT-LoRA r=32", "#8B5CF6", "-.")):
    hp = OUT_MULTI_ROOT / folder / "history.csv"
    if hp.exists():
        dh = pd.read_csv(hp); lw = 2.6 if folder == "r8" else 1.7
        axes[0].plot(dh["epoch"], dh["val_loss"], label=lbl, color=col, ls=ls, lw=lw, marker="o", ms=3.5)
        axes[1].plot(dh["epoch"], dh["val_f1"] * 100, label=lbl, color=col, ls=ls, lw=lw, marker="s", ms=3.5)
axes[0].set_title("(a) Validation Loss theo Epoch (Đa lớp)", fontweight="bold"); axes[0].set_xlabel("Epoch"); axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.4)
axes[1].set_title("(b) Validation Macro F1 theo Epoch (Đa lớp)", fontweight="bold"); axes[1].set_xlabel("Epoch"); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.4)
plt.tight_layout()
save_fig_all(fig, "fig4_ablation_epoch_convergence_curves.png")

for fn in ("fig1_ablation_multiclass_rank_study.png", "fig2_ablation_binary_rank_study.png", "fig4_ablation_epoch_convergence_curves.png"):
    ipy_display(Image(filename=str(DRIVE_OUT / "figures" / fn), width=850))

---
## Cell 10: Xuất Báo cáo `BAO_CAO_TUAN3.md` & Đóng gói `.zip` Tải về

In [ ]:
def df_to_md(df: pd.DataFrame) -> str:
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join(str(r[c]) for c in cols) + " |")
    return "\n".join(lines)

md_text = (
    "# BÁO CÁO TUẦN 3 — Nguyễn Thanh Hải (Dataset 3: BRISC 2025)\n\n"
    "## 1. Ablation Study số 1 — Phân loại Đa lớp (4 Lớp)\n\n"
    + df_to_md(df_ablation_multi) + "\n\n"
    "## 2. Ablation Study số 1 — Phân loại Nhị phân (Có u / Không u)\n\n"
    + df_to_md(df_ablation_bin) + "\n"
)
(DRIVE_OUT / "BAO_CAO_TUAN3.md").write_text(md_text, encoding="utf-8")
print(md_text)

zip_path = DRIVE_OUT / "week3_hai_brisc2025_ablation_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in DRIVE_OUT.rglob("*"):
        if p.is_file() and p != zip_path and p.suffix.lower() not in {".pt", ".pth", ".zip"}:
            zf.write(p, p.relative_to(DRIVE_OUT))

print(f"\n Đóng gói thành công: {zip_path} ({zip_path.stat().st_size / 1e6:.2f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))